# 05. Python data processing, ETL, data quality, scipy tests and a random forest

This is the Python data processing section. The notebook does the full pandas ETL on the 9 raw CSVs, runs a data quality scorecard before and after cleaning, applies the same zone canonicalisation as the MongoDB and SQL pipelines, builds an analytical wide table, runs scipy chi square tests that pair with the R analytics, and fits a scikit-learn random forest classifier with cross validation.

## Setup

In [ ]:
!pip install -q pandas numpy scipy matplotlib scikit-learn

import pandas as pd
import numpy as np

# The 9 NorthStar CSVs are committed to the GitHub repo, so the notebook
# pulls them straight over HTTPS rather than asking the marker to mount
# Drive or upload anything.
BASE_URL = "https://raw.githubusercontent.com/D1lxrry/Nortstar-Task/main/northstar_dataset"


## Stage 1. Extract

In [ ]:
CSV_FILES = ["customers", "orders", "deliveries", "drivers", "vehicles",
             "hubs", "incidents", "complaints", "app_events"]
frames = {name: pd.read_csv(f"{BASE_URL}/{name}.csv") for name in CSV_FILES}
for name, df in frames.items():
    print(f"{name:<12} {df.shape[0]:>5} rows  {df.shape[1]:>2} cols")


## Stage 2 + 4. Data quality scorecard

In [ ]:
SCHEMA = {
    "customers":  ("customer_id",  []),
    "orders":     ("order_id",     [("customer_id", "customers")]),
    "deliveries": ("delivery_id",  [("order_id", "orders"), ("driver_id", "drivers"),
                                    ("vehicle_id", "vehicles"), ("hub_id", "hubs")]),
    "drivers":    ("driver_id",   []),
    "vehicles":   ("vehicle_id",  []),
    "hubs":       ("hub_id",      []),
    "incidents":  ("incident_id",  [("delivery_id", "deliveries")]),
    "complaints": ("complaint_id", [("order_id", "orders"), ("customer_id", "customers")]),
    "app_events": ("event_id",    [("customer_id", "customers"), ("order_id", "orders")]),
}
ZONE_FIELDS_BY_TABLE = {
    "customers": ["home_zone"], "orders": ["pickup_zone", "dropoff_zone"],
    "drivers": ["base_zone"], "vehicles": ["assigned_zone"],
    "hubs": ["zone"], "app_events": ["zone_context"],
}

def quality_report(frames, stage):
    rows = []
    for table, (pk, fks) in SCHEMA.items():
        df = frames[table]
        zones = ZONE_FIELDS_BY_TABLE.get(table, [])
        zone_distinct = sum(df[z].nunique(dropna=True) for z in zones if z in df.columns)
        fk_violations = 0
        for fk_col, ref_table in fks:
            if fk_col in df.columns:
                ref_keys = set(frames[ref_table][SCHEMA[ref_table][0]].dropna().unique())
                fk_violations += int(df[fk_col].dropna().apply(lambda v, k=ref_keys: v not in k).sum())
        rows.append({
            "stage": stage, "table": table, "rows": len(df),
            "null_cells": int(df.isna().sum().sum()),
            "duplicate_pk": int(df[pk].duplicated().sum()) if pk in df.columns else None,
            "fk_violations": fk_violations,
            "zone_distinct_total": zone_distinct,
        })
    return pd.DataFrame(rows)

print("BEFORE cleanup")
pre = quality_report(frames, "before")
print(pre.to_string(index=False))

## Stage 3. Transform

In [ ]:
ZONE_MAP = {
    "AIRPORT": "Airport", "Airport": "Airport",
    "CENTRAL": "Central", "Central": "Central", "Ctr": "Central",
    "EAST": "East", "East": "East",
    "NORTH": "North", "North": "North", "north": "North",
    "RiverSide": "Riverside", "Riverside": "Riverside",
    "SOUTH": "South", "South": "South",
    "WEST": "West", "West": "West",
}

def canon_zone(s): return s.astype("object").map(lambda v: ZONE_MAP.get(v, v) if pd.notna(v) else v)

for table, df in frames.items():
    for col in df.columns:
        lower = col.lower()
        if any(k in lower for k in ("_at", "_time", "_date", "timestamp")):
            df[col] = pd.to_datetime(df[col], errors="coerce")
    for zone_col in ZONE_FIELDS_BY_TABLE.get(table, []):
        if zone_col in df.columns:
            df[zone_col] = canon_zone(df[zone_col])
    frames[table] = df

print("AFTER cleanup")
post = quality_report(frames, "after")
print(post.to_string(index=False))

## Build the analytical wide table

In [ ]:
orders = frames["orders"].copy()
complaint_counts = frames["complaints"].groupby("order_id").size().rename("complaint_count")
event_counts = frames["app_events"].dropna(subset=["order_id"]).groupby("order_id").size().rename("app_event_count")
incident_counts = frames["incidents"].groupby("delivery_id").size().rename("incident_count")

deliveries = frames["deliveries"].merge(incident_counts, left_on="delivery_id", right_index=True, how="left")
deliveries["incident_count"] = deliveries["incident_count"].fillna(0).astype(int)

dat = (orders
       .merge(deliveries, on="order_id", how="left", suffixes=("", "_delivery"))
       .merge(frames["customers"], on="customer_id", how="left", suffixes=("", "_cust"))
       .merge(complaint_counts, left_on="order_id", right_index=True, how="left")
       .merge(event_counts, left_on="order_id", right_index=True, how="left"))
for col in ["complaint_count", "app_event_count", "incident_count"]:
    dat[col] = dat[col].fillna(0).astype(int)
dat["has_delivery"] = dat["delivery_id"].notna()
dat["delivered"] = dat["delivery_status"].isin(["OnTime", "Delayed", "Failed"])
dat["failed"] = dat["delivery_status"].eq("Failed")

print(f"Analytical table: {dat.shape}")

## scipy chi square tests

In [ ]:
from scipy import stats

delivered = dat[dat["delivered"]].copy()

tab1 = pd.crosstab(delivered["pickup_zone"], delivered["delivery_status"])
chi2, p, dof, _ = stats.chi2_contingency(tab1)
print(f"P1. pickup_zone vs delivery_status:  X^2 = {chi2:.2f}, df = {dof}, p = {p:.4f}")

tab2 = pd.crosstab(delivered["service_type"], delivered["delivery_status"])
chi2, p, dof, _ = stats.chi2_contingency(tab2)
print(f"P2. service_type vs delivery_status: X^2 = {chi2:.2f}, df = {dof}, p = {p:.4f}")

## scikit-learn random forest classifier

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

feats_cat = ["service_type", "pickup_zone", "priority_level"]
feats_num = ["route_distance_km", "loyalty_score", "app_engagement_score",
             "incident_count", "complaint_count", "app_event_count"]

mdl_data = dat[dat["delivered"]].dropna(subset=feats_cat + feats_num + ["failed"]).copy()
mdl_data["failed"] = mdl_data["failed"].astype(int)
print(f"n = {len(mdl_data)}, class balance = {mdl_data['failed'].mean():.3f}")

X, y = mdl_data[feats_cat + feats_num], mdl_data["failed"]
pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), feats_cat),
    ("num", StandardScaler(), feats_num),
])
pipe = Pipeline([("preprocess", pre),
                 ("clf", RandomForestClassifier(
                     n_estimators=400, min_samples_leaf=4,
                     class_weight="balanced", random_state=42, n_jobs=-1))])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
auc = cross_val_score(pipe, X, y, cv=cv, scoring="roc_auc", n_jobs=-1)
print(f"5 fold CV ROC AUC: {auc.mean():.3f} +/- {auc.std():.3f}  folds={np.round(auc, 3)}")

In [ ]:
# Hold out evaluation, classification report, feature importance.
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
pipe.fit(X_tr, y_tr)
y_proba = pipe.predict_proba(X_te)[:, 1]
y_pred = pipe.predict(X_te)
print(f"Hold out ROC AUC: {roc_auc_score(y_te, y_proba):.3f}")
print(classification_report(y_te, y_pred, target_names=["not failed", "failed"]))

feature_names = list(pipe.named_steps["preprocess"].named_transformers_["cat"]
                     .get_feature_names_out(feats_cat)) + feats_num
fi = pd.DataFrame({"feature": feature_names,
                   "importance": pipe.named_steps["clf"].feature_importances_})
fi.sort_values("importance", ascending=False).head(10)

## What the Python run showed

The chi square on `pickup_zone` came out at X^2 = 26.12, df = 12, p = 0.0103, which is the same value the R analytics produced, so scipy and R agree to 4 decimal places.

The random forest with 9 candidate predictors only reached 5 fold CV ROC AUC of around 0.49, which is no better than guessing. That lines up with the near null logistic regression fit in notebook 04. So the operational signal lives at the zone level rather than at the per order level, and the practical recommendation for NorthStar is to fix the zones that fail most rather than try to flag individual risky orders.